# Solutions — Diagnostic 1 — Core Python

Exercise notebook: `00_diagnostic/diagnostic_01_core_python.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Question 1 — 🟢 Easy · Parse a duration

**Working solution**

In [ ]:
import re

_DURATION = re.compile(r"(?:(\d+)h)?(?:(\d+)m)?(?:(\d+)s)?")

def parse_duration(text):
    match = _DURATION.fullmatch(text.strip().lower())
    if not match or not any(match.groups()):
        raise ValueError(f"invalid duration: {text!r}")
    h, m, s = (int(g) if g else 0 for g in match.groups())
    return h * 3600 + m * 60 + s

**Explanation**

One anchored regex with optional groups validates order and extracts parts. `fullmatch` rejects trailing garbage; checking `any(groups)` rejects the empty match.

**Complexity:** O(n).

**Common mistakes**

- Scanning character by character with a state machine you don't need.
- Accepting `'30s1h'`.

## Question 2 — 🟡 Medium · Validate IPv4 addresses

**Working solution**

In [ ]:
def is_ipv4(text):
    parts = text.split(".")
    if len(parts) != 4:
        return False
    for p in parts:
        if not p.isdigit() or not p.isascii() or (len(p) > 1 and p[0] == "0") or int(p) > 255:
            return False
    return True

**Explanation**

Split, then validate each piece with guard clauses. `str.isdigit()` accepts non-ASCII digits like `'²'`, hence the `isascii()` check. (`ipaddress.ip_address` exists in the stdlib — also a fine answer if you know its strictness rules.)

**Complexity:** O(n).

**Common mistakes**

- `int(p)` on `'+1'` or `' 1'` succeeds — validate the text first.

## Question 3 — 🟡 Medium · Compare version strings

**Working solution**

In [ ]:
from itertools import zip_longest

def _parts(v):
    v = v.strip().lstrip("vV")
    try:
        return [int(x) for x in v.split(".")]
    except ValueError:
        raise ValueError(f"invalid version: {v!r}") from None

def compare_versions(a, b):
    for x, y in zip_longest(_parts(a), _parts(b), fillvalue=0):
        if x != y:
            return 1 if x > y else -1
    return 0

**Explanation**

`zip_longest(..., fillvalue=0)` pads the shorter version. (Lists of ints also compare lexicographically: padding both and comparing lists works too.)

**Complexity:** O(parts).

**Common mistakes**

- Comparing strings: `'1.10' < '1.9'` lexicographically.

## Question 4 — 🟠 Hard · Word-wrap for log messages

**Working solution**

In [ ]:
def wrap(text, width):
    if width < 1:
        raise ValueError("width must be >= 1")
    lines, current = [], ""
    for word in text.split():
        pieces = [word[i:i + width] for i in range(0, len(word), width)]
        for piece in pieces:
            if not current:
                current = piece
            elif len(current) + 1 + len(piece) <= width:
                current += " " + piece
            else:
                lines.append(current)
                current = piece
    if current:
        lines.append(current)
    return lines

**Explanation**

Greedy line filling with one 'current line' accumulator. (`textwrap.wrap(text, width, break_long_words=True)` is the stdlib answer — knowing it exists is worth points too.)

**Complexity:** O(n).

**Common mistakes**

- Forgetting the final line.
- Off-by-one on the separating space.